# pystreme smoke test

The literal Goal-1 check from `DESIGNDOC.md`: de novo motif discovery from a notebook cell, no subprocess, no FASTA round trip. This stays a manual notebook (not a pytest test) because the point is to confirm the *ergonomics*, not just the numeric output.

Run it from a GPU allocation (a GPU node, or a notebook session running on one); on CPU the `fit` cell takes minutes rather than seconds. Set the paths below to your own genome (indexed FASTA), summit-centred peaks and a MEME-format motif database.


In [ ]:
import os, itertools, torch
from pystreme import MotifDiscovery, summary, annotate
from pystreme import plot

GENOME = "path/to/genome.fa"       # indexed FASTA (.fai alongside)
PEAKS = "path/to/peaks.bed"         # summit-centred peaks
HOCOMOCO = "path/to/H12CORE_meme_format.meme"  # any MEME-format motif database works
device = "cuda" if torch.cuda.is_available() else "cpu"

# first 2000 peaks, to keep this quick
peaks = "smoke_peaks.bed"
with open(PEAKS) as src, open(peaks, "w") as dst:
    dst.writelines(itertools.islice(src, 2000))
device

## Build the sequence universe and run discovery

`from_bed` extracts `size` bp around each peak center (HOMER `-size` semantics) and fits an order-2 background. `fit` is the STREME-style round loop: dinucleotide-shuffled control, 10% hold-out, one motif per round, erasing between rounds.

In [ ]:
disc = MotifDiscovery.from_bed(peaks, GENOME, size=200, device=device, seed=0)
motifs = disc.fit(n_motifs=3, verbose=True)
motifs

## Results as tables

`holdout_logp` is the significance to quote (the training threshold re-tested on peaks the search never saw); `train_logp` is the selected, optimistic number. `central_logp` tells directly bound (sharply central) from co-occurring motifs.

In [ ]:
summary(motifs)

In [ ]:
sites = motifs[0].sites_frame()   # one row per peak: best site, genomic coordinates, strand, passing
sites[sites.passing].head()

## Name what was found

In-process Tomtom-style ranking (sum of per-column Pearson correlations over the best alignment) against a motif database. Expect the SP1 GC-box first, then NF-Y (CCAAT) and AP-1 on this peak set.

In [ ]:
if os.path.exists(HOCOMOCO):
    ann = annotate(motifs, HOCOMOCO, top=3)
    display(ann)

## Logos and positional distributions

In [ ]:
fig = plot.report(motifs, annotation=ann if os.path.exists(HOCOMOCO) else None)  # most significant first

## Export for Tomtom / FIMO / other tools

In [ ]:
disc.to_meme(motifs, "smoke_motifs.meme")
disc.to_bed(motifs, "smoke_sites.bed")     # BED6 of every passing site, in genome coordinates
print(open("smoke_motifs.meme").read()[:600])

## Scanning a known motif

The scanner and the SEA-style enrichment test are independently useful: score a JASPAR/HOCOMOCO PWM against the peaks, or test it against a matched control.

In [ ]:
from pystreme.meme_io import read_meme
sp1 = read_meme("../tests/fixtures/MA0079.1_SP1.meme")["MA0079.1"]
hit = disc.scan(sp1.pwm)                      # best score / position / strand per peak
enr = disc.enrichment(sp1.pwm)                # optimal threshold + Fisher p vs dinucleotide-shuffled control
float(hit.scores.mean()), enr